In [1]:
import anndata
import scanpy as sc
import pandas as pd
import cell2sentence as cs
from cell2sentence.tasks import embed_cells
import os
from pathlib import Path
import logging
import tracemalloc
import linecache

ModuleNotFoundError: No module named 'cell2sentence'

In [2]:
ref = anndata.read_h5ad("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/GSE220608/GSM6807287_cleanso.h5ad")
gene_name1 = ref.var["gene_name"]

In [3]:
file1 = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/cell_lines/train/signature_matrix_for_nnls.csv"
c2s_save_dir1 = "/gpfs/commons/groups/compbio/projects/CZI_endom/RNA_temp/c2s_deconv"
c2s_save_name1 = "cell_line_signature"
model_path1 = "/gpfs/commons/groups/compbio/projects/rf_projects/rf_packages/C2S-Pythia-410m-cell-type-prediction"
model_save_dir1 = "/gpfs/commons/groups/compbio/projects/CZI_endom/RNA_temp/c2s_deconv"
model_save_name1 = "C2S-Pythia-410m-cell-type-prediction"

In [16]:
def display_top(snapshot, key_type='lineno', limit=10):
    snapshot = snapshot.filter_traces((
        tracemalloc.Filter(False, "<frozen importlib._bootstrap>"),
        tracemalloc.Filter(False, "<unknown>"),
    ))
    top_stats = snapshot.statistics(key_type)

    print("Top %s lines" % limit)
    for index, stat in enumerate(top_stats[:limit], 1):
        frame = stat.traceback[0]
        print("#%s: %s:%s: %.1f KiB"
              % (index, frame.filename, frame.lineno, stat.size / 1024))
        line = linecache.getline(frame.filename, frame.lineno).strip()
        if line:
            print('    %s' % line)

    other = top_stats[limit:]
    if other:
        size = sum(stat.size for stat in other)
        print("%s other: %.1f KiB" % (len(other), size / 1024))
    total = sum(stat.size for stat in top_stats)
    print("Total allocated size: %.1f KiB" % (total / 1024))
    return(total / 1024 / 1024)


In [19]:
def get_embedding_c2s(file,
                      c2s_save_dir, c2s_save_name, 
                      model_path, model_save_dir, model_save_name,
                      embedding_save_dir = None,
                      index_name = "Unnamed: 0", gene_name = None, reorder_obs_name = None, 
                      log = True, log_path = None):
    tracemalloc.start()
    # Load data
    if log:
        logging.basicConfig(format='%(asctime)s %(message)s', 
                            filename=os.path.dirname(file1) + "/c2s_embeddings_" + Path(file1).stem + ".log",
                            filemode='w', force = True)
        logging.warning('Prep data')
    data = pd.read_csv(file, header = 0)
    data = data.set_index(index_name)
    adata = sc.AnnData(data)
    if gene_name is not None:
        adata.var_names = gene_name
    adata.obs["organism"]="Homo sapiens"
    adata.obs["cell_type"]="unknown"
    adata.obs["tissue"]="unknown"
    adata.obs["sex"]="unknown"
    adata.obs["batch_condition"]="unknown"
    adata_obs_cols_to_keep = ["organism","cell_type","tissue", "sex", "batch_condition"]

    # Create CSData object
    if log:
        logging.warning('Prep model')
    arrow_ds, vocabulary = cs.CSData.adata_to_arrow(
        adata=adata, 
        random_state=42, 
        sentence_delimiter=' ',
        label_col_names=adata_obs_cols_to_keep
    )
    csdata = cs.CSData.csdata_from_arrow(
        arrow_dataset=arrow_ds, 
        vocabulary=vocabulary,
        save_dir=c2s_save_dir,
        save_name=c2s_save_name,
        dataset_backend="arrow"
    )

    # Define CSModel object
    csmodel = cs.CSModel(
        model_name_or_path=model_path,
        save_dir=model_save_dir,
        save_name=model_save_name
    )

    # Get embeddings
    if log:
        logging.warning('Extract embedding')
    embedded_cells = embed_cells(
        csdata=csdata,
        csmodel=csmodel,
        n_genes=200
    )
    df = pd.DataFrame(embedded_cells)
    if reorder_obs_name is None:
        df["name"] = adata.obs_names.to_list()
    else:
        df["name"] = ["sample_" + str(x) for x in list(range(1, df.shape[0]+1))]
    if log:
        logging.warning('Save embedding')
    if embedding_save_dir is None: 
        df.set_index('name').to_csv(os.path.dirname(file1) + "/c2s_embeddings_" + Path(file1).stem + ".csv")
    else:
        df.set_index('name').to_csv(embedding_save_dir + "/c2s_embeddings_" + c2s_save_name + ".csv")
    if log:
        logging.warning('Done')
    snapshot = tracemalloc.take_snapshot()
    mem=display_top(snapshot)
    logging.warning('Memory used: ' + str(mem) + " MiB")


In [20]:
get_embedding_c2s(file1,
                  c2s_save_dir1, c2s_save_name1, 
                  model_path1, model_save_dir1, model_save_name1,
                  gene_name = gene_name1, index_name = "type")

WARN: more variables (36581) than observations (6)... did you mean to transpose the object (e.g. adata.T)?
WARN: more variables (36581) than observations (6), did you mean to transpose the object (e.g. adata.T)?
Saving the dataset (1/1 shards): 100%|███| 6/6 [00:00<00:00, 1041.20 examples/s]


Using device: cuda
Reloading model from path on disk: /gpfs/commons/groups/compbio/projects/CZI_endom/RNA_temp/c2s_deconv/C2S-Pythia-410m-cell-type-prediction
Embedding 6 cells using CSModel...


100%|█████████████████████████████████████████████| 6/6 [00:00<00:00, 48.42it/s]


Top 10 lines
#1: /gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/cell2sentence/utils.py:60: 17423.5 KiB
    vocabulary[name.upper()] = gene_sums[i]  # keys are all uppercase gene names
#2: /gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/pandas/core/internals/blocks.py:540: 6859.9 KiB
    values = values.copy()
#3: /gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/pandas/io/parsers/c_parser_wrapper.py:93: 6859.1 KiB
    self._reader = parsers.TextReader(src, **kwds)
#4: /gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/logging/__init__.py:1176: 4096.7 KiB
    return open(self.baseFilename, self.mode, encoding=self.encoding)
#5: /gpfs/commons/groups/compbio/projects/rf_condas/cell2sentence/lib/python3.8/site-packages/pandas/core/indexes/base.py:2238: 3098.0 KiB
    return self._engine.is_unique
#6: /gpfs/commons/groups/compbio/projects/rf_c